# Flight Data Processing Notebook

This notebook processes raw flight data from the bronze table, performs data cleaning and feature engineering, and writes the results to a silver table. It also computes origin-level metrics such as total flights, delays, cancellation rates, and peak departure hours.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, hour, format_string, date_format
from pyspark.sql import functions as F

bronze_table = "task11.core.bronze_flights"

df = spark.table(bronze_table)

Converts the string columns into the respective data types for processing.

In [0]:
df = df.withColumn("fl_date", to_date(col("fl_date"), "M/d/yyyy"))
df = df.withColumn(
    "flight_date",
    to_date("fl_date", "M/d/yyyy")
)
int_cols = [
    "year", "month", "day_of_month", "day_of_week",
    "dep_time", "taxi_out", "wheels_off", "wheels_on",
    "taxi_in", "air_time", "distance", "weather_delay",
    "late_aircraft_delay"
]

for c in int_cols:
    df = df.withColumn(c, col(c).cast("int"))

Converts the time columns from "1247" to "12:47" or HH:MM format from the bronze layer to insert into the silver layer table.

In [0]:
from pyspark.sql.functions import when, col, format_string

def convert_time(colname):
    return when(
        col(colname).isNull(), None
    ).otherwise(
        format_string(
            "%02d:%02d",
            (col(colname) / 100).cast("int"),
            (col(colname) % 100).cast("int")
        )
    )

df = df.withColumn("dep_time_str", convert_time("dep_time")) \
       .withColumn("wheels_off_str", convert_time("wheels_off")) \
       .withColumn("wheels_on_str", convert_time("wheels_on"))


The below function converts the three columns from normal time to new time stamp columns to be inserted into the silver layer table.

In [0]:
from pyspark.sql import functions as F

def safe_timestamp(date_col, time_col):
    return F.when(time_col.isNull(), None) \
            .when(time_col == "24:00",
                  F.to_timestamp(F.concat(F.date_add(date_col, 1), F.lit(" 00:00")), "yyyy-MM-dd HH:mm")
                 ).otherwise(
                  F.to_timestamp(F.concat(date_col, F.lit(" "), time_col), "yyyy-MM-dd HH:mm")
                 )


df = df.withColumn("dep_timestamp", safe_timestamp(col("flight_date"), col("dep_time_str"))) \
       .withColumn("wheels_off_ts", safe_timestamp(col("flight_date"), col("wheels_off_str"))) \
       .withColumn("wheels_on_ts", safe_timestamp(col("flight_date"), col("wheels_on_str")))


Converts the cancelled field from 0/1 to boolean.

In [0]:
df = df.withColumn("cancelled", col("cancelled").cast("boolean"))

Calculates if any delay was incurred by the aircraft and the gate-to-gate time. 

In [0]:
df = df.withColumn(
    "gate_to_gate_time",
    (col("wheels_on_ts").cast("long") - col("wheels_off_ts").cast("long")) / 60
)

df = df.withColumn("dep_hour", hour(col("dep_timestamp")))

df = df.withColumn(
    "weather_delay",
    col("weather_delay").cast("int")
).withColumn(
    "late_aircraft_delay",
    col("late_aircraft_delay").cast("int")
)

df = df.withColumn(
    "total_delay",
    col("weather_delay") + col("late_aircraft_delay")
)

df = df.withColumn("is_delayed", (col("total_delay") > 0))


Write the new columns and transformed data to the silver table.

In [0]:
silver_path = "/Volumes/task11/core/silver_flights"

df.write.format("delta").mode("overwrite").saveAsTable("task11.core.silver_flights")


Checks if the expected values occur in the table after writing.

In [0]:
%sql
DESCRIBE TABLE task11.core.silver_flights;

SELECT * FROM task11.core.silver_flights LIMIT 10;

year,month,day_of_month,day_of_week,fl_date,origin,origin_city_name,origin_state_nm,dep_time,taxi_out,wheels_off,wheels_on,taxi_in,cancelled,air_time,distance,weather_delay,late_aircraft_delay,flight_date,dep_time_str,wheels_off_str,wheels_on_str,dep_timestamp,wheels_off_ts,wheels_on_ts,gate_to_gate_time,dep_hour,total_delay,is_delayed
2024,1,1,1,2024-01-01,JFK,"New York, NY",New York,1247,31,1318,1442,7,false,84,509,0,0,2024-01-01,12:47,13:18,14:42,2024-01-01T12:47:00.000Z,2024-01-01T13:18:00.000Z,2024-01-01T14:42:00.000Z,84.0,12,0,false
2024,1,1,1,2024-01-01,MSP,"Minneapolis, MN",Minnesota,1001,20,1021,1249,6,false,88,622,0,0,2024-01-01,10:01,10:21,12:49,2024-01-01T10:01:00.000Z,2024-01-01T10:21:00.000Z,2024-01-01T12:49:00.000Z,148.0,10,0,false
2024,1,1,1,2024-01-01,JFK,"New York, NY",New York,1411,21,1432,1533,8,false,61,288,0,0,2024-01-01,14:11,14:32,15:33,2024-01-01T14:11:00.000Z,2024-01-01T14:32:00.000Z,2024-01-01T15:33:00.000Z,61.0,14,0,false
2024,1,1,1,2024-01-01,RIC,"Richmond, VA",Virginia,1643,13,1656,1747,12,false,51,288,0,0,2024-01-01,16:43,16:56,17:47,2024-01-01T16:43:00.000Z,2024-01-01T16:56:00.000Z,2024-01-01T17:47:00.000Z,51.0,16,0,false
2024,1,1,1,2024-01-01,DTW,"Detroit, MI",Michigan,1010,21,1031,1016,4,false,45,237,0,0,2024-01-01,10:10,10:31,10:16,2024-01-01T10:10:00.000Z,2024-01-01T10:31:00.000Z,2024-01-01T10:16:00.000Z,-15.0,10,0,false
2024,1,1,1,2024-01-01,JAX,"Jacksonville, FL",Florida,1403,14,1417,1559,4,false,102,833,0,0,2024-01-01,14:03,14:17,15:59,2024-01-01T14:03:00.000Z,2024-01-01T14:17:00.000Z,2024-01-01T15:59:00.000Z,102.0,14,0,false
2024,1,1,1,2024-01-01,LGA,"New York, NY",New York,947,26,1013,1218,13,false,125,833,0,0,2024-01-01,09:47,10:13,12:18,2024-01-01T09:47:00.000Z,2024-01-01T10:13:00.000Z,2024-01-01T12:18:00.000Z,125.0,9,0,false
2024,1,1,1,2024-01-01,CHS,"Charleston, SC",South Carolina,1135,8,1143,1309,5,false,86,641,0,0,2024-01-01,11:35,11:43,13:09,2024-01-01T11:35:00.000Z,2024-01-01T11:43:00.000Z,2024-01-01T13:09:00.000Z,86.0,11,0,false
2024,1,1,1,2024-01-01,LGA,"New York, NY",New York,810,14,824,1005,8,false,101,641,0,0,2024-01-01,08:10,08:24,10:05,2024-01-01T08:10:00.000Z,2024-01-01T08:24:00.000Z,2024-01-01T10:05:00.000Z,101.0,8,0,false
2024,1,1,1,2024-01-01,ITH,"Ithaca/Cortland, NY",New York,1248,12,1300,1343,12,false,43,189,0,0,2024-01-01,12:48,13:00,13:43,2024-01-01T12:48:00.000Z,2024-01-01T13:00:00.000Z,2024-01-01T13:43:00.000Z,43.0,12,0,false


The cell below calculates the origin metric for all the airports and will be later used to narrow down the metrics and decide which airport is the best performing and which is the worst performing airport. 
These metrics will also be helpful in calculating the weekly flights fact table.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

origin_metrics = (
    df.groupBy("origin")
    .agg(
        F.count("*").alias("total_flights"),
        F.sum(F.col("is_delayed").cast("int")).alias("delayed_flights"),
        F.avg("total_delay").alias("avg_delay"),
        (F.sum(F.col("cancelled").cast("double")) / F.count("*")).alias("cancellation_rate")
    )
)

peak_hour_df = (
    df.groupBy("origin", "dep_hour").count()
)
window = Window.partitionBy("origin").orderBy(F.desc("count"))
peak_hour_df = (
    peak_hour_df.withColumn("rank", F.row_number().over(window))
    .filter(F.col("rank")==1)
    .select("origin", F.col("dep_hour").alias("peak_departure_hour"))
)

origin_metrics = origin_metrics.join(peak_hour_df, on="origin", how="left")
origin_metrics.write.format("delta").mode("overwrite").saveAsTable("task11.core.origin_metrics")
